In [ ]:
import uproot
import pandas as pd
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

# Read NuSyst Tree ROOT file

In [ ]:
with uproot.open("NuSystTree.root") as f:

    df_event = f["events"].arrays(library="pd")
    df_meta = f["tweak_metadata"].arrays(library="pd")

# Let's first look at the metadata tree

In [ ]:
df_meta

Metadata contains the dial and and its list of variations used when running `DumpConfiguredTweaksNuSyst`.
- Each row is for each dial
- "fString": "Full" name of the dial
  - <tool_type>_<instance_name>_<dial name>
- "ntweaks": Number of variations
- "Tweakvalues": Array of the variation

# Now let's look at the event tree

In [ ]:
df_event.head(5)

Event tree converts GENIE Event info into a simple flat tree.
Each row corresponds to a single GENIE interaction.
It contains various kinematic variables that desecribes the interaction.

For instance, we have neutrino energy in "Enu_true":

In [ ]:
df_event["Enu_true"].hist(bins=np.linspace(0, 60, 60+1))

Above should be the same as what you see from the morning session!

Now, let's focus on the last few columns, which contains the reweight information

In [ ]:
DialColumnName_prefix = "DUNEDAS2026ExampleReweighter_NuSystTutorial"

DialColNames = [
    f"ntweaks_{DialColumnName_prefix}_DialA",
    f"tweak_responses_{DialColumnName_prefix}_DialA",
    f"paramCVWeight_{DialColumnName_prefix}_DialA",
]

df_event[DialColNames].head(5)

- "ntweaks_(prefix)_(DialName)"
    - Number of variations for the Dial
    - Same as in the metadata
- "tweak_responses_(prefix)_(DialName)"
    - An array of the reweights for each variation
    - Reweight in this Tree is divided by the reweight of the "central_value"
    - The order of the elements are the same as "tweakvalues" from the metadata
- "paramCVWeight_(prefix)_(DialName)"
    - Reweight for the central_value
    - If the reweight from cental_value is not 1.0, this means we want to apply a "CV" correction using this dial, and the reweights are on top of that correction
    - For this tutorial, let's make RW(central_value) to be 1.0

As you can see, the reweights from the current module is all 1.0, so we stay at the CV.

Now let's modify "DUNEDAS2026ExampleReweighter" module!